# Build Networks Test Set CSV

Creates a tidy CSV mapping raw network labels/abbreviations to canonical NeuroVLM network keys. The output includes the raw label, canonical key, mapped network name, mapped region terms, mapped cognitive terms, and a concise definition.

In [ ]:
from pathlib import Path
import re
import pandas as pd

In [ ]:
NETWORK_CANONICAL = {
    "visual": {
        "display": "Visual",
        "aliases": ["visual", "vision", "vis", "primary visual", "extrastriate", "occipital"],
        "short_definition": "Visual network for visual perception, object recognition, scene processing, motion perception, and visuospatial processing.",
        "long_definition": "The visual network includes primary and extrastriate visual cortices, lateral occipital cortex, fusiform gyrus, ventral occipitotemporal cortex, middle temporal visual area, lateral temporal-occipital cortex, intraparietal visual regions, and dorsal occipital-parietal areas. It supports visual perception, object and face recognition, scene processing, motion perception, visuospatial processing, and visually guided attention.",
    },
    "motor": {
        "display": "Motor",
        "aliases": ["motor", "somatomotor", "sensorimotor", "smot", "somatosensory", "movement", "premotor"],
        "short_definition": "Sensorimotor network for movement planning, voluntary movement execution, proprioception, touch, and sensorimotor integration.",
        "long_definition": "The motor or somatomotor network includes primary motor cortex, primary somatosensory cortex, premotor cortex, supplementary motor area, paracentral lobule, cerebellum, basal ganglia, and thalamus. It supports voluntary movement execution, motor planning, touch, pain, temperature, proprioception, sensorimotor integration, motor learning, and motor coordination.",
    },
    "auditory": {
        "display": "Auditory",
        "aliases": ["auditory", "aud", "hearing", "acoustic", "speech-sound", "superior temporal"],
        "short_definition": "Auditory network for auditory perception, sound localization, pitch processing, music perception, and speech-sound processing.",
        "long_definition": "The auditory network includes primary auditory cortex, planum temporale, planum polare, superior temporal gyrus, superior temporal sulcus, and insula. It supports auditory perception, pitch processing, sound localization, speech-sound processing, music perception, and detection of salient acoustic events.",
    },
    "language": {
        "display": "Language",
        "aliases": ["language", "lang", "speech", "semantic", "syntax", "reading", "phonological"],
        "short_definition": "Language network for speech comprehension, speech production, syntax, semantics, reading, lexical access, and phonological processing.",
        "long_definition": "The language network includes inferior frontal gyrus, posterior superior temporal gyrus, middle temporal gyrus, anterior temporal lobe, angular gyrus, and supramarginal gyrus. It supports speech comprehension and production, syntax processing, semantic processing, reading, lexical access, phonological processing, and sentence-level language integration.",
    },
    "attention": {
        "display": "Attention",
        "aliases": ["attention", "dorsal attention", "dattn", "dorsattn", "spatial orienting", "frontal eye fields"],
        "short_definition": "Dorsal attention network for selective attention, spatial orienting, sustained attention, and top-down attentional control.",
        "long_definition": "The attention network emphasizes dorsal attention regions including intraparietal sulcus, dorsal superior parietal lobule, frontal eye fields, superior frontal cortex, temporoparietal junction, ventral frontal cortex, precuneus, and parietal midline. It supports selective attention, spatial orienting, sustained attention, top-down attentional control, and bottom-up reorienting to behaviorally relevant stimuli.",
    },
    "frontoparietal_control": {
        "display": "Frontoparietal Control",
        "aliases": ["frontoparietal", "control", "fpn", "executive", "working memory", "multiple demand", "cognitive control"],
        "short_definition": "Frontoparietal control network for executive control, working memory, planning, task switching, goal representation, and adaptive control.",
        "long_definition": "The frontoparietal control network includes dorsolateral prefrontal cortex, inferior frontal sulcus, lateral prefrontal cortex, anterior prefrontal cortex, inferior parietal lobule, intraparietal sulcus, and lateral temporal cortex. It supports executive control, working memory, cognitive flexibility, task switching, rule and goal representation, decision making, planning, and adaptive control.",
    },
    "cingulo_opercular": {
        "display": "Cingulo-Opercular",
        "aliases": ["cingulo", "opercular", "cingulo-opercular", "salience", "salventattn", "insula", "error detection", "performance monitoring"],
        "short_definition": "Cingulo-opercular/salience network for salience detection, performance monitoring, stable task-set maintenance, interoception, and sustained control.",
        "long_definition": "The cingulo-opercular network includes dorsal anterior cingulate cortex, medial superior frontal cortex, anterior insula, frontal operculum, anterior prefrontal cortex, anterior midline regions, and thalamus. It supports salience detection, performance monitoring, error detection, stable task-set maintenance, sustained control, autonomic coordination, and interoception.",
    },
    "default_mode": {
        "display": "Default Mode",
        "aliases": ["default", "default mode", "dmn", "dn", "autobiographical", "self-referential", "mind-wandering", "theory of mind"],
        "short_definition": "Default mode network for self-referential thought, autobiographical memory, future thinking, spontaneous thought, and social cognition.",
        "long_definition": "The default mode network includes medial prefrontal cortex, posterior cingulate cortex, precuneus, angular gyrus, lateral inferior parietal cortex, lateral temporal cortex, retrosplenial cortex, posterior midline regions, and hippocampal formation. It supports self-referential thought, mind-wandering, spontaneous thought, autobiographical memory, future thinking, social cognition, and theory of mind.",
    },
}
DISPLAY_TO_KEY = {v["display"]: k for k, v in NETWORK_CANONICAL.items()}
KEY_TO_DISPLAY = {k: v["display"] for k, v in NETWORK_CANONICAL.items()}

In [ ]:
regions_map = {
    "Visual": ["Primary visual cortex", "Extrastriate visual cortex", "Lateral occipital cortex", "Fusiform gyrus", "Ventral occipitotemporal cortex", "Middle temporal visual area", "Lateral temporal-occipital", "Intraparietal visual", "Dorsal occipital-parietal"],
    "Motor": ["Primary motor cortex", "Primary somatosensory cortex", "Premotor cortex", "Supplementary motor area", "Paracentral lobule", "Cerebellum", "Basal ganglia", "Thalamus"],
    "Auditory": ["Primary auditory cortex", "Planum temporale", "Planum polare", "Superior temporal gyrus", "Superior temporal sulcus", "Insula"],
    "Language": ["Inferior frontal gyrus", "Posterior superior temporal gyrus", "Middle temporal gyrus", "Anterior temporal lobe", "Angular gyrus", "Supramarginal gyrus"],
    "Attention": ["Intraparietal sulcus", "Superior parietal lobule (dorsal)", "Frontal eye fields", "Superior frontal", "Temporoparietal junction", "Ventral frontal cortex", "Precuneus", "Parietal midline"],
    "Frontoparietal Control": ["Dorsolateral prefrontal cortex", "Inferior frontal sulcus", "Lateral PFC", "Anterior prefrontal", "Inferior parietal lobule", "Intraparietal sulcus", "Lateral temporal cortex"],
    "Cingulo-Opercular": ["Dorsal anterior cingulate cortex (dACC)", "Medial superior frontal cortex", "Anterior insula", "Frontal operculum", "Anterior prefrontal", "Anterior midline", "Thalamus"],
    "Default Mode": ["Medial prefrontal cortex (mPFC)", "Posterior cingulate cortex", "Precuneus", "Angular gyrus", "Lateral inferior parietal cortex", "Lateral temporal cortex", "Retrosplenial", "Posterior midline regions", "Hippocampal formation"],
    "Limbic Affective": ["Orbitofrontal cortex (medial & lateral OFC)", "Ventromedial prefrontal cortex (vmPFC)", "Temporal pole", "Anterior temporal cortex", "Amygdala", "Ventral striatum", "Nucleus accumbens", "Subgenual", "Ventral anterior cingulate"],
    "Medial Temporal Lobe": ["Hippocampus", "Entorhinal cortex", "Parahippocampal cortex", "Perirhinal cortex", "Retrosplenial cortex", "Posterior cingulate cortex", "Posterior medial regions"],
}

cognition_map = {
    "Visual": ["Visual perception", "Object recognition", "Face processing", "Scene processing", "Motion perception", "Visuospatial processing", "Visual attention"],
    "Somatomotor": ["Voluntary movement execution", "Motor planning", "Somatosensory processing", "Touch", "Pain", "Temperature", "Proprioception", "Sensorimotor integration", "Motor learning", "Motor coordination"],
    "Auditory": ["Auditory perception", "Pitch processing", "Sound localization", "Speech-sound processing", "Music perception"],
    "Language": ["Speech comprehension", "Speech production", "Syntax processing", "Semantic processing", "Reading", "Lexical access", "Phonological processing"],
    "Attention": ["Selective attention", "Spatial orienting", "Sustained attention", "Top-down attentional control", "Bottom-up reorienting"],
    "Frontoparietal Control": ["Executive control", "Working memory", "Cognitive flexibility", "Task switching", "Rule/goal representation", "Decision making", "Planning", "Adaptive control"],
    "Cingulo-Opercular": ["Salience detection", "Performance monitoring", "Error detection", "Stable task-set maintenance", "Sustained control", "Autonomic coordination", "Interoception"],
    "Default Mode": ["Self-referential thought", "Mind-wandering", "Spontaneous thought", "Autobiographical memory", "Future thinking", "Social cognition", "Theory of mind"],
    "Limbic Affective": ["Emotion processing", "Affect regulation", "Reward valuation", "Motivation", "Approach-avoidance", "Social-emotional appraisal"],
    "Medial Temporal Lobe": ["Episodic memory", "Encoding", "Retrieval", "Relational memory", "Associative memory", "Context binding", "Spatial navigation", "Memory consolidation"],
}

In [ ]:
label_map = [
 ('VIS-P', 'unknown'), ('CG-OP', 'cingulo_opercular'), ('DN-B', 'default_mode'), ('SMOT-B', 'motor'), ('AUD', 'auditory'), ('PM-PPr', 'motor'), ('dATN-B', 'attention'), ('SMOT-A', 'motor'), ('LANG', 'language'), ('FPN-B', 'frontoparietal_control'), ('FPN-A', 'frontoparietal_control'), ('dATN-A', 'attention'), ('VIS-C', 'visual'), ('SAL/PMN', 'cingulo_opercular'), ('DN-A', 'default_mode'), ('NONE', 'unknown'), ('Visual1', 'visual'), ('Visual2', 'visual'), ('Somatomotor', 'motor'), ('CingOperc', 'cingulo_opercular'), ('DorsAttn', 'attention'), ('Language', 'language'), ('FrontPar', 'frontoparietal_control'), ('Auditory', 'auditory'), ('Default', 'default_mode'), ('PostMulti', 'unknown'), ('VentMulti', 'unknown'), ('OrbitAffective', 'unknown'), ('Emo/Interoception1', 'unknown'), ('Emo/Interoception2', 'unknown'), ('Emo/Interoception3', 'unknown'), ('Emo/Interoception4', 'unknown'), ('Mot/Visspatial1', 'motor'), ('Mot/Visspatial2', 'motor'), ('Mot/Visspatial3', 'motor'), ('Mot/Visspatial4', 'motor'), ('Visual3', 'visual'), ('DivergentCog1', 'unknown'), ('DivergentCog3', 'unknown'), ('DivergentCog4', 'unknown'), ('DivergentCog5', 'unknown'), ('DivergentCog6', 'unknown'), ('medial frontal', 'frontoparietal_control'), ('frontoparietal', 'frontoparietal_control'), ('default mode', 'default_mode'), ('motor cortex', 'motor'), ('visual A', 'visual'), ('visual B', 'visual'), ('visual association', 'visual'), ('subcortical cerebellum', 'unknown'), ('AntSal', 'cingulo_opercular'), ('DorsalDMN', 'default_mode'), ('HighVisual', 'visual'), ('LECN', 'frontoparietal_control'), ('PostSal', 'cingulo_opercular'), ('Precuneus', 'default_mode'), ('PrimVisual', 'visual'), ('RECN', 'frontoparietal_control'), ('Sensorimotor', 'motor'), ('VentralDMN', 'default_mode'), ('Visuospatial', 'unknown'), ('LatVis', 'visual'), ('MedVis', 'visual'), ('Premotor', 'motor'), ('Salience', 'cingulo_opercular'), ('HandSM', 'motor'), ('FaceSM', 'motor'), ('AntMTL', 'unknown'), ('PostMTL', 'unknown'), ('ParMemory', 'unknown'), ('Context', 'unknown'), ('FootSM', 'motor'), ('Visual', 'visual'), ('VentAttn', 'unknown'), ('DorsalSM', 'motor'), ('VentralSM', 'motor'), ('MedPar', 'default_mode'), ('ParOcc', 'visual'), ('SCAN', 'frontoparietal_control'), ('Cingulo-Opercular', 'cingulo_opercular'), ('Effector-hand', 'motor'), ('Effector-mouth', 'motor'), ('Effector-foot', 'motor'), ('SM', 'motor'), ('LateralSM', 'motor'), ('ResponseOneHanded(1RESP)', 'motor'), ('ResponseTwoHanded(2RESP)', 'motor'), ('AuditoryAttentionResponse(AAR)', 'auditory'), ('AuditoryPrimarySensory(AUD)', 'auditory'), ('DMNNovel(DMNA)', 'default_mode'), ('DMNTraditional(DMNB)', 'default_mode'), ('FocusOnVisualFeatures(FoVF)', 'visual'), ('Initiation(INIT)', 'unknown'), ('Language(LN)', 'language'), ('MAIN', 'unknown'), ('MultipleDemand(MDN)', 'frontoparietal_control'), ('Re-evaluation(RE-EV)', 'frontoparietal_control'), ('DefaultA', 'default_mode'), ('DefaultB', 'default_mode'), ('DefaultC', 'default_mode'), ('ContA', 'frontoparietal_control'), ('ContB', 'frontoparietal_control'), ('ContC', 'frontoparietal_control'), ('SalVenAttnA', 'cingulo_opercular'), ('SalVenAttnB', 'cingulo_opercular'), ('DorsAttnA', 'attention'), ('DorsAttnB', 'attention'), ('Aud', 'auditory'), ('SomMotA', 'motor'), ('SomMotB', 'motor'), ('VisualA', 'visual'), ('VisualB', 'visual'), ('VisualC', 'visual'), ('TempPar', 'frontoparietal_control'), ('LimbicA', 'unknown'), ('LimbicB', 'unknown'), ('SalVentAttnB', 'cingulo_opercular'), ('SalVentAttnA', 'cingulo_opercular'), ('VisPeri', 'visual'), ('VisCent', 'visual'), ('SomatomotorA', 'motor'), ('SomatomotorB', 'motor'), ('Sal/VenAttnA', 'cingulo_opercular'), ('Sal/VenAttnB', 'cingulo_opercular'), ('ControlC', 'frontoparietal_control'), ('ControlA', 'frontoparietal_control'), ('ControlB', 'frontoparietal_control'), ('Sal/VenAttn', 'cingulo_opercular'), ('Limbic', 'unknown'), ('Control', 'frontoparietal_control')]

In [ ]:
def canonical_display(network_key):
    return NETWORK_CANONICAL.get(network_key, {}).get("display", "Unknown")


def terms_for_key(network_key):
    display = canonical_display(network_key)
    cognition_display = "Somatomotor" if network_key == "motor" else display
    spec = NETWORK_CANONICAL.get(network_key, {})
    return {
        "region_terms": "; ".join(regions_map.get(display, [])),
        "cognitive_terms": "; ".join(cognition_map.get(cognition_display, [])),
        "short_definition": spec.get("short_definition", "Unknown or intentionally excluded network label."),
        "long_definition": spec.get("long_definition", "Unknown or intentionally excluded network label."),
    }

rows = []
seen = set()
for raw_label, network_key in label_map:
    key = (raw_label, network_key)
    if key in seen:
        continue
    seen.add(key)
    term_info = terms_for_key(network_key)
    mapped_terms = []
    if network_key != "unknown":
        mapped_terms = [canonical_display(network_key)] + NETWORK_CANONICAL[network_key]["aliases"]
    rows.append({
        "raw_network_label": raw_label,
        "network_key": network_key,
        "network_name": canonical_display(network_key),
        "mapped_terms": "; ".join(dict.fromkeys(mapped_terms)),
        **term_info,
    })

network_test_set = pd.DataFrame(rows).sort_values(["network_key", "raw_network_label"]).reset_index(drop=True)
output_path = Path("docs/03_evaluation/network_test_set_labels.csv")
network_test_set.to_csv(output_path, index=False)
print(f"Wrote {len(network_test_set)} rows to {output_path}")
display(network_test_set.head(20))